# 01 - Análise Exploratória de Dados

Objetivo deste notebook: importar, ler e entender o conjunto de dados Instacart Market Basket Analysis antes de qualquer modelagem.

Nesta etapa vamos responder:

- Do que se trata a base de dados?
- Quais arquivos, colunas e relacionamentos existem?
- Qual é a qualidade inicial das variáveis?
- Existem nulos, duplicados ou problemas de integridade?
- Quais sinais parecem úteis para um sistema de recomendação?

## 1. Configuração inicial

Nesta etapa são carregadas as bibliotecas utilizadas no notebook e definidas opções de exibição do pandas. Também é criado o caminho base para localizar os arquivos brutos do Instacart, mantendo o notebook executável tanto a partir da pasta `notebooks` quanto da raiz do projeto.

In [3]:
from collections import Counter, defaultdict
from itertools import combinations
from pathlib import Path
import sqlite3

import numpy as np
import pandas as pd

# Configura a exibição para facilitar a leitura das tabelas no notebook.
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", "{:.2f}".format)

### Definição dos caminhos

O caminho dos dados brutos é centralizado em `RAW_DATA_DIR`. Essa definição evita caminhos fixos espalhados pelo notebook e facilita a reprodução da análise em outro ambiente.

In [5]:
# Define caminhos de forma flexível para rodar o notebook da raiz ou da pasta notebooks.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw" / "instacart"

RAW_DATA_DIR

PosixPath('/Users/cassiojr/FIAP/Tech Challenge 2/FIAP-fase2-e-commerce/data/raw/instacart')

## 2. Arquivos disponíveis

A base é composta por tabelas relacionais. Os arquivos maiores são os itens comprados em pedidos anteriores e a tabela de pedidos.

In [5]:
# Lista os arquivos brutos e seus tamanhos para entender o volume inicial da base.
csv_files = sorted(RAW_DATA_DIR.glob("*.csv"))

file_inventory = pd.DataFrame(
    {
        "file_name": [file.name for file in csv_files],
        "size_mb": [file.stat().st_size / 1024**2 for file in csv_files],
    }
).sort_values("size_mb", ascending=False)

file_inventory

,file_name,size_mb
2,order_products__prior.csv,550.80
4,orders.csv,103.92
3,order_products__train.csv,23.54
5,products.csv,2.07
0,aisles.csv,0.00
1,departments.csv,0.00


## 3. Leitura dos dados

Os arquivos CSV são carregados em DataFrames separados, respeitando a estrutura original da base. Em seguida, eles são agrupados em um dicionário para facilitar comparações de volume, memória, tipos de dados e qualidade.

In [23]:
# Carrega cada tabela da base Instacart em um DataFrame separado.
aisles = pd.read_csv(RAW_DATA_DIR / "aisles.csv")
departments = pd.read_csv(RAW_DATA_DIR / "departments.csv")
products = pd.read_csv(RAW_DATA_DIR / "products.csv")
orders = pd.read_csv(RAW_DATA_DIR / "orders.csv")
order_products_train = pd.read_csv(RAW_DATA_DIR / "order_products__train.csv")
order_products_prior = pd.read_csv(RAW_DATA_DIR / "order_products__prior.csv")

# Agrupa os DataFrames para facilitar análises repetidas de volume e qualidade.
datasets = {
    "aisles": aisles,
    "departments": departments,
    "products": products,
    "orders": orders,
    "order_products_train": order_products_train,
    "order_products_prior": order_products_prior,
}

### Visão geral dos volumes

A tabela abaixo resume o tamanho de cada DataFrame em número de linhas, colunas e memória utilizada. Esse passo ajuda a identificar quais tabelas exigem mais atenção em joins, agregações e etapas futuras de modelagem.

In [24]:
overview = pd.DataFrame(
    {
        "dataset": name,
        "rows": len(df),
        "columns": df.shape[1],
        "memory_mb": df.memory_usage(deep=True).sum() / 1024**2,
    }
    for name, df in datasets.items()
).sort_values("rows", ascending=False)

overview

,dataset,rows,columns,memory_mb
5,order_products_prior,32434489,4,989.82
3,orders,3421083,7,332.71
4,order_products_train,1384617,4,42.26
2,products,49688,4,4.93
0,aisles,134,2,0.01
1,departments,21,2,0.00


## 4. Amostra e esquema dos dados

Nesta parte vamos observar colunas, tipos de dados e primeiras linhas para entender a função de cada tabela.

In [25]:
def describe_schema(dataframes: dict[str, pd.DataFrame]) -> pd.DataFrame:
    """Resume colunas, tipos e quantidade de valores nulos por dataset."""
    rows = []

    for dataset_name, df in dataframes.items():
        for column in df.columns:
            rows.append(
                {
                    "dataset": dataset_name,
                    "column": column,
                    "dtype": str(df[column].dtype),
                    "nulls": df[column].isna().sum(),
                    "null_rate": df[column].isna().mean(),
                    "unique_values": df[column].nunique(dropna=True),
                }
            )

    return pd.DataFrame(rows)


schema = describe_schema(datasets)
schema

,dataset,column,dtype,nulls,null_rate,unique_values
0,aisles,aisle_id,int64,0,0.00,134
1,aisles,aisle,str,0,0.00,134
2,departments,department_id,int64,0,0.00,21
3,departments,department,str,0,0.00,21
4,products,product_id,int64,0,0.00,49688
5,products,product_name,str,0,0.00,49688
6,products,aisle_id,int64,0,0.00,134
7,products,department_id,int64,0,0.00,21
8,orders,order_id,int64,0,0.00,3421083
9,orders,user_id,int64,0,0.00,206209


### Inspeção visual das tabelas

A exibição das primeiras linhas complementa o resumo do esquema dos dados. Aqui é possível conferir o formato real dos registros, validar nomes de colunas e entender como as chaves se conectam entre as tabelas.

In [ ]:
for name, df in datasets.items():
    print(f"\n{name}")
    display(df.head())

## 5. O que cada tabela representa

- `orders`: pedidos por usuário, com ordem temporal, dia da semana, hora e dias desde o pedido anterior.
- `order_products__prior`: itens de pedidos anteriores. Esta tabela representa o histórico principal de comportamento.
- `order_products__train`: itens de pedidos no conjunto de treino. Pode ser usada como alvo de validação offline.
- `products`: catálogo de produtos, com ligação para `aisles` e `departments`.
- `aisles`: corredores/categorias intermediárias do catálogo.
- `departments`: departamentos principais do catálogo.

Para recomendação, o sinal central é a interação `user_id -> product_id`, obtida ao juntar `orders` com as tabelas `order_products`.

## 6. Qualidade dos dados

Vamos verificar nulos, duplicados e valores fora do domínio esperado.

### Valores ausentes

A primeira verificação de qualidade mede a quantidade e a proporção de valores nulos por coluna. Esse diagnóstico separa ausências esperadas, como o primeiro pedido de cada usuário, de possíveis problemas que precisariam de tratamento.

In [ ]:
missing_summary = (
    schema.loc[schema["nulls"] > 0]
    .sort_values(["null_rate", "nulls"], ascending=False)
    .reset_index(drop=True)
)

missing_summary

### Registros duplicados

A checagem de duplicidade confirma se há linhas repetidas nos arquivos carregados. Duplicatas poderiam distorcer contagens de compra, taxas de recompra e métricas de popularidade dos produtos.

In [ ]:
duplicate_summary = pd.DataFrame(
    {
        "dataset": name,
        "duplicate_rows": df.duplicated().sum(),
        "duplicate_rate": df.duplicated().mean(),
    }
    for name, df in datasets.items()
)

duplicate_summary

### Validação de domínio

Além de nulos e duplicados, algumas colunas possuem faixas de valores esperadas. Esta validação verifica dias da semana, horários, ordem dos pedidos, posição no carrinho e a variável binária de recompra.

In [ ]:
domain_checks = {
    "orders_invalid_order_dow": orders.loc[~orders["order_dow"].between(0, 6)].shape[0],
    "orders_invalid_order_hour": orders.loc[~orders["order_hour_of_day"].between(0, 23)].shape[0],
    "orders_invalid_order_number": orders.loc[orders["order_number"] < 1].shape[0],
    "prior_invalid_add_to_cart_order": order_products_prior.loc[order_products_prior["add_to_cart_order"] < 1].shape[0],
    "train_invalid_add_to_cart_order": order_products_train.loc[order_products_train["add_to_cart_order"] < 1].shape[0],
    "prior_invalid_reordered": order_products_prior.loc[~order_products_prior["reordered"].isin([0, 1])].shape[0],
    "train_invalid_reordered": order_products_train.loc[~order_products_train["reordered"].isin([0, 1])].shape[0],
}

pd.Series(domain_checks, name="invalid_rows").to_frame()

Observação esperada: `days_since_prior_order` deve possuir nulos no primeiro pedido de cada usuário, porque não existe pedido anterior.

In [ ]:
first_orders = orders["order_number"].eq(1)

pd.DataFrame(
    {
        "scenario": ["first_order", "not_first_order"],
        "rows": [first_orders.sum(), (~first_orders).sum()],
        "days_since_prior_order_nulls": [
            orders.loc[first_orders, "days_since_prior_order"].isna().sum(),
            orders.loc[~first_orders, "days_since_prior_order"].isna().sum(),
        ],
    }
)

## 7. Integridade relacional

Aqui validamos se as chaves de uma tabela encontram correspondência nas tabelas de referência.

### Validação das chaves

Como a base é relacional, é importante garantir que produtos, pedidos, corredores e departamentos estejam corretamente referenciados. Falhas nessa etapa poderiam gerar perdas ou inconsistências durante os joins.

In [26]:
integrity_checks = {
    "products_without_aisle": (~products["aisle_id"].isin(aisles["aisle_id"])).sum(),
    "products_without_department": (~products["department_id"].isin(departments["department_id"])).sum(),
    "prior_items_without_order": (~order_products_prior["order_id"].isin(orders["order_id"])).sum(),
    "train_items_without_order": (~order_products_train["order_id"].isin(orders["order_id"])).sum(),
    "prior_items_without_product": (~order_products_prior["product_id"].isin(products["product_id"])).sum(),
    "train_items_without_product": (~order_products_train["product_id"].isin(products["product_id"])).sum(),
}

pd.Series(integrity_checks, name="invalid_references").to_frame()

,invalid_references
products_without_aisle,0
products_without_department,0
prior_items_without_order,0
train_items_without_order,0
prior_items_without_product,0
train_items_without_product,0


## 8. Entendimento das variáveis principais

Com a qualidade e a integridade verificadas, a análise passa a observar as variáveis mais importantes para recomendação: sequência dos pedidos, horário da compra, intervalo entre compras e divisão entre conjuntos de avaliação.

### Estatísticas dos pedidos

As estatísticas descritivas resumem o comportamento temporal dos pedidos. Elas ajudam a observar frequência de compra, distribuição dos horários e possíveis limites naturais das variáveis.

In [27]:
orders[["order_number", "order_dow", "order_hour_of_day", "days_since_prior_order"]].describe()

,order_number,order_dow,order_hour_of_day,days_since_prior_order
count,3421083.00,3421083.00,3421083.00,3214874.00
mean,17.15,2.78,13.45,11.11
std,17.73,2.05,4.23,9.21
min,1.00,0.00,0.00,0.00
25%,5.00,1.00,10.00,4.00
50%,11.00,3.00,13.00,7.00
75%,23.00,5.00,16.00,15.00
max,100.00,6.00,23.00,30.00


In [28]:
orders["eval_set"].value_counts(normalize=True).rename("rate").to_frame().join(
    orders["eval_set"].value_counts().rename("rows")
)

,rate,rows
eval_set,,
prior,0.94,3214874
train,0.04,131209
test,0.02,75000


### Catálogo enriquecido

O catálogo de produtos é enriquecido com corredor e departamento. Essa visão torna as análises mais interpretáveis, pois substitui parte dos identificadores numéricos por categorias de negócio.

In [29]:
product_catalog = (
    products.merge(aisles, on="aisle_id", how="left")
    .merge(departments, on="department_id", how="left")
)

product_catalog.head()

,product_id,product_name,aisle_id,department_id,aisle,department
0,1,Chocolate Sandwich Cookies,61,19,cookies cakes,snacks
1,2,All-Seasons Salt,104,13,spices seasonings,pantry
2,3,Robust Golden Unsweetened Oolong Tea,94,7,tea,beverages
3,4,Smart Ones Classic Favorites Mini Rigatoni With Vodka Cream Sauce,38,1,frozen meals,frozen
4,5,Green Chile Anytime Sauce,5,13,marinades meat preparation,pantry


In [30]:
department_product_counts = (
    product_catalog["department"]
    .value_counts()
    .rename_axis("department")
    .reset_index(name="products")
)

department_product_counts.head(15)

,department,products
0,personal care,6563
1,snacks,6264
2,pantry,5371
3,beverages,4365
4,frozen,4007
5,dairy eggs,3449
6,household,3085
7,canned goods,2092
8,dry goods pasta,1858
9,produce,1684


## 9. Sinais comportamentais para recomendação

O objetivo do projeto é recomendar produtos com base no comportamento do usuário, não apenas prever recompra. Por isso, a análise passa a observar seis perguntas principais:

- Quais produtos o usuário já comprou?
- Quais categorias ele costuma consumir?
- Quais produtos aparecem juntos nos mesmos pedidos?
- Quais produtos usuários parecidos compram?
- Quais produtos são populares dentro dos interesses dele?
- Quais itens ele ainda não comprou, mas têm alta afinidade com seu histórico?

### Construção das interações históricas

Unimos itens comprados, pedidos e catálogo para formar a tabela central de comportamento. Cada linha representa um produto comprado por um usuário em um pedido específico.

In [ ]:
# Une itens comprados, contexto do pedido e catálogo de produtos.
prior_interactions = (
    order_products_prior
    .merge(
        orders[["order_id", "user_id", "order_number", "order_dow", "order_hour_of_day", "days_since_prior_order"]],
        on="order_id",
        how="left",
    )
    .merge(product_catalog, on="product_id", how="left")
)

prior_interactions.head()

### Produtos que o usuário já comprou

O histórico individual mostra preferências explícitas do usuário. Ele é útil como sinal de afinidade, mas não deve ser a única fonte de candidatos, pois isso limitaria o sistema a recomendações de recompra.

In [ ]:
# Escolhe um usuário com bastante histórico para ilustrar as análises comportamentais.
example_user_id = orders["user_id"].value_counts().idxmax()

# Resume quais produtos esse usuário comprou mais vezes.
user_purchase_history = (
    prior_interactions.loc[prior_interactions["user_id"].eq(example_user_id)]
    .groupby(["product_id", "product_name", "aisle", "department"], as_index=False)
    .agg(
        purchase_count=("order_id", "nunique"),
        avg_cart_position=("add_to_cart_order", "mean"),
        last_order_number=("order_number", "max"),
    )
    .sort_values(["purchase_count", "last_order_number"], ascending=False)
)

user_purchase_history.head(15)

### Categorias que o usuário costuma consumir

As categorias favoritas ajudam a recomendar produtos que ainda não foram comprados, mas pertencem a departamentos e corredores com forte presença no histórico do usuário.

In [ ]:
# Agrupa o histórico do usuário por departamento e corredor.
user_category_preferences = (
    prior_interactions.loc[prior_interactions["user_id"].eq(example_user_id)]
    .groupby(["department", "aisle"], as_index=False)
    .agg(items_purchased=("product_id", "count"), unique_products=("product_id", "nunique"))
    .sort_values("items_purchased", ascending=False)
)

user_category_preferences.head(15)

### Produtos que aparecem juntos nos mesmos pedidos

A coocorrência identifica produtos frequentemente comprados no mesmo carrinho. Esse sinal permite sugerir itens complementares, mesmo que o usuário ainda não tenha comprado o produto recomendado.

In [ ]:
# Usa uma amostra de pedidos para estimar produtos comprados juntos sem sobrecarregar a memória.
COOC_SAMPLE_ORDERS = 50_000
TOP_RELATED_PER_PRODUCT = 8

sampled_order_ids = (
    order_products_prior["order_id"]
    .drop_duplicates()
    .sample(n=min(COOC_SAMPLE_ORDERS, order_products_prior["order_id"].nunique()), random_state=42)
)
cooc_source = order_products_prior.loc[
    order_products_prior["order_id"].isin(sampled_order_ids),
    ["order_id", "product_id"],
]

# Conta pares de produtos que aparecem no mesmo pedido.
pair_counts = Counter()
for _, products_in_order in cooc_source.groupby("order_id")["product_id"]:
    basket = sorted(set(products_in_order))

    # Ignora carrinhos muito pequenos ou muito grandes para reduzir ruído e custo.
    if len(basket) < 2 or len(basket) > 30:
        continue

    for left, right in combinations(basket, 2):
        pair_counts[(left, right)] += 1
        pair_counts[(right, left)] += 1

# Junta os IDs ao catálogo para tornar o resultado interpretável.
cooccurrence_sample = (
    pd.DataFrame(
        [(left, right, count) for (left, right), count in pair_counts.items()],
        columns=["product_id", "related_product_id", "cooccurrence_count"],
    )
    .sort_values("cooccurrence_count", ascending=False)
    .head(20)
    .merge(product_catalog[["product_id", "product_name"]], on="product_id", how="left")
    .merge(
        product_catalog[["product_id", "product_name"]].rename(
            columns={"product_id": "related_product_id", "product_name": "related_product_name"}
        ),
        on="related_product_id",
        how="left",
    )
)

cooccurrence_sample

### Produtos comprados por usuários parecidos

Uma forma simples de aproximar usuários semelhantes é comparar suas categorias favoritas. Usuários com o mesmo corredor favorito tendem a compartilhar interesses de consumo.

In [ ]:
# Identifica o corredor favorito de cada usuário pelo número de itens comprados.
user_aisle_counts = (
    prior_interactions.groupby(["user_id", "aisle_id", "aisle"], as_index=False)
    .agg(items_purchased=("product_id", "count"))
)

favorite_aisle_by_user = (
    user_aisle_counts.sort_values(["user_id", "items_purchased"], ascending=[True, False])
    .drop_duplicates("user_id")
    .rename(columns={"aisle_id": "favorite_aisle_id", "aisle": "favorite_aisle"})
    [["user_id", "favorite_aisle_id", "favorite_aisle"]]
)

# Usa o corredor favorito como aproximação simples de usuários semelhantes.
example_favorite_aisle = favorite_aisle_by_user.loc[
    favorite_aisle_by_user["user_id"].eq(example_user_id),
    "favorite_aisle_id",
].iat[0]

similar_user_products = (
    prior_interactions.merge(favorite_aisle_by_user[["user_id", "favorite_aisle_id"]], on="user_id", how="left")
    .query("favorite_aisle_id == @example_favorite_aisle")
    .groupby(["product_id", "product_name", "aisle", "department"], as_index=False)
    .agg(similar_users=("user_id", "nunique"), orders_with_product=("order_id", "nunique"))
    .sort_values(["similar_users", "orders_with_product"], ascending=False)
)

similar_user_products.head(15)

### Produtos populares dentro dos interesses do usuário

Também podemos recomendar produtos populares nos departamentos e corredores favoritos do usuário. Esse sinal amplia a descoberta sem sair das preferências observadas.

In [ ]:
# Seleciona os departamentos favoritos do usuário de exemplo.
example_favorite_departments = (
    prior_interactions.loc[prior_interactions["user_id"].eq(example_user_id)]
    .groupby("department_id")
    .size()
    .sort_values(ascending=False)
    .head(2)
    .index
)

# Busca produtos populares dentro dessas categorias de interesse.
popular_inside_user_interests = (
    prior_interactions.loc[prior_interactions["department_id"].isin(example_favorite_departments)]
    .groupby(["product_id", "product_name", "aisle", "department"], as_index=False)
    .agg(orders_with_product=("order_id", "nunique"), unique_users=("user_id", "nunique"))
    .sort_values(["unique_users", "orders_with_product"], ascending=False)
)

popular_inside_user_interests.head(15)

### Itens novos com alta afinidade

Por fim, removemos produtos já comprados pelo usuário e observamos quais itens novos aparecem com maior força nos sinais de coocorrência, usuários parecidos e categorias favoritas.

In [ ]:
# Remove produtos já comprados para destacar candidatos novos para o usuário.
already_purchased = set(user_purchase_history["product_id"])

new_affinity_candidates = (
    pd.concat(
        [
            similar_user_products.assign(source="similar_users"),
            popular_inside_user_interests.assign(source="favorite_categories"),
        ],
        ignore_index=True,
    )
    .loc[lambda df: ~df["product_id"].isin(already_purchased)]
    .drop_duplicates("product_id")
    .head(20)
)

new_affinity_candidates[["source", "product_id", "product_name", "aisle", "department"]].head(15)

## 10. Construção da nova base para modelagem

A nova base de treino combina quatro fontes comportamentais de candidatos:

- histórico do usuário;
- produtos coocorrentes no mesmo pedido;
- produtos populares nas categorias favoritas;
- produtos comprados por usuários com preferência semelhante.

Dessa forma, a base passa a conter tanto produtos já comprados quanto produtos novos com afinidade comportamental.

### Parâmetros da geração de candidatos

Os limites abaixo controlam o tamanho da base. Eles evitam criar todas as combinações possíveis entre usuários e produtos, o que seria muito grande e pouco útil para o protótipo.

In [ ]:
# Configura a saída SQLite e os limites usados na geração de candidatos.
DATABASE_PATH = PROJECT_ROOT / "data" / "training_data.db"
TABLE_NAME = "training_data"
NEW_TABLE_NAME = "training_data_new"

RANDOM_STATE = 42
COOC_SAMPLE_ORDERS = 250_000
TOP_RELATED_PER_PRODUCT = 8
USER_SEED_PRODUCTS = 5
USER_COOCCURRENCE_CANDIDATES = 12
USER_FAVORITE_DEPARTMENTS = 2
USER_FAVORITE_AISLES = 2
CATEGORY_TOP_PRODUCTS = 10
USER_CATEGORY_CANDIDATES = 12
SIMILAR_TOP_PRODUCTS = 10

### Features históricas e de categoria

Criamos variáveis por usuário-produto, por usuário, por produto e por preferência de categoria. Para produtos novos, as variáveis históricas do par usuário-produto serão preenchidas com zero.

In [ ]:
# Base principal: cada linha representa um produto comprado em um pedido anterior.
dataset_all_orders = order_products_prior.merge(
    orders[["order_id", "user_id", "order_number", "order_dow", "order_hour_of_day", "days_since_prior_order"]],
    on="order_id",
    how="left",
).merge(products[["product_id", "aisle_id", "department_id"]], on="product_id", how="left")

# O conjunto train define quais produtos aparecem na próxima compra de cada usuário.
train_user_products = (
    order_products_train.merge(orders[["order_id", "user_id"]], on="order_id", how="left")
    [["user_id", "product_id"]]
    .drop_duplicates()
)

# Features do par usuário-produto, calculadas apenas com histórico anterior.
user_product_features = (
    dataset_all_orders.groupby(["user_id", "product_id"], as_index=False)
    .agg(
        purchase_count=("order_id", "count"),
        reorder_rate=("reordered", "mean"),
        avg_cart_position=("add_to_cart_order", "mean"),
        first_order_number=("order_number", "min"),
        last_order_number=("order_number", "max"),
    )
)

# Features gerais do comportamento de compra do usuário.
user_features = (
    dataset_all_orders.groupby("user_id", as_index=False)
    .agg(
        user_total_items=("product_id", "count"),
        user_unique_products=("product_id", "nunique"),
        user_reorder_rate=("reordered", "mean"),
        user_avg_cart_position=("add_to_cart_order", "mean"),
    )
)

# Features temporais do usuário extraídas da tabela de pedidos.
user_order_features = (
    orders.groupby("user_id", as_index=False)
    .agg(
        user_total_orders=("order_number", "max"),
        user_avg_days_between_orders=("days_since_prior_order", "mean"),
        user_avg_order_hour=("order_hour_of_day", "mean"),
    )
)

user_features = user_features.merge(user_order_features, on="user_id", how="left")
user_features["user_avg_basket_size"] = user_features["user_total_items"] / user_features["user_total_orders"].clip(lower=1)

# Features globais de popularidade e recompra do produto.
product_features = (
    dataset_all_orders.groupby("product_id", as_index=False)
    .agg(
        product_total_orders=("order_id", "count"),
        product_unique_users=("user_id", "nunique"),
        product_reorder_rate=("reordered", "mean"),
        product_avg_cart_position=("add_to_cart_order", "mean"),
    )
)

In [ ]:
# Mede quanto cada usuário consome em cada departamento.
user_department_counts = (
    dataset_all_orders.groupby(["user_id", "department_id"], as_index=False)
    .agg(user_department_purchase_count=("product_id", "count"))
    .merge(user_features[["user_id", "user_total_items"]], on="user_id", how="left")
)
user_department_counts["user_department_purchase_rate"] = (
    user_department_counts["user_department_purchase_count"] / user_department_counts["user_total_items"].clip(lower=1)
)
user_department_counts = user_department_counts.drop(columns="user_total_items")

# Mede quanto cada usuário consome em cada corredor.
user_aisle_counts = (
    dataset_all_orders.groupby(["user_id", "aisle_id"], as_index=False)
    .agg(user_aisle_purchase_count=("product_id", "count"))
    .merge(user_features[["user_id", "user_total_items"]], on="user_id", how="left")
)
user_aisle_counts["user_aisle_purchase_rate"] = (
    user_aisle_counts["user_aisle_purchase_count"] / user_aisle_counts["user_total_items"].clip(lower=1)
)
user_aisle_counts = user_aisle_counts.drop(columns="user_total_items")

# Ordena departamentos e corredores por preferência de cada usuário.
favorite_departments = (
    user_department_counts.sort_values(["user_id", "user_department_purchase_count"], ascending=[True, False])
    .assign(rank=lambda df: df.groupby("user_id").cumcount() + 1)
)
favorite_aisles = (
    user_aisle_counts.sort_values(["user_id", "user_aisle_purchase_count"], ascending=[True, False])
    .assign(rank=lambda df: df.groupby("user_id").cumcount() + 1)
)

# Guarda a categoria favorita principal para usar em features e usuários semelhantes.
user_profile = (
    favorite_departments.query("rank == 1")[["user_id", "department_id"]]
    .rename(columns={"department_id": "favorite_department_id"})
    .merge(
        favorite_aisles.query("rank == 1")[["user_id", "aisle_id"]].rename(columns={"aisle_id": "favorite_aisle_id"}),
        on="user_id",
        how="outer",
    )
)

### Geração das fontes de candidatos

Cada fonte gera pares `user_id` e `product_id` com flags específicas. Depois, os pares são consolidados para que um produto possa ter mais de uma razão para ser recomendado.

In [ ]:
def with_source_flags(df: pd.DataFrame, source: str) -> pd.DataFrame:
    """Adiciona flags de origem para um conjunto de candidatos."""
    result = df.copy()

    # Inicializa todas as origens como ausentes.
    for column in [
        "candidate_from_history",
        "candidate_from_cooccurrence",
        "candidate_from_favorite_category",
        "candidate_from_similar_users",
    ]:
        result[column] = 0

    # Marca somente a origem recebida pela função.
    result[f"candidate_from_{source}"] = 1

    # Garante que todos os candidatos tenham as mesmas colunas de score.
    for score_column in ["cooccurrence_score", "category_popularity_score", "similar_user_score"]:
        if score_column not in result.columns:
            result[score_column] = 0.0

    return result

# Produtos já comprados pelo usuário são uma das fontes de candidatos.
history_candidates = with_source_flags(user_product_features[["user_id", "product_id"]], "history")

In [ ]:
# Amostra pedidos para calcular coocorrência sem gerar todos os pares da base completa.
sampled_order_ids = (
    order_products_prior["order_id"]
    .drop_duplicates()
    .sample(n=min(COOC_SAMPLE_ORDERS, order_products_prior["order_id"].nunique()), random_state=RANDOM_STATE)
)
cooc_source = order_products_prior.loc[
    order_products_prior["order_id"].isin(sampled_order_ids),
    ["order_id", "product_id"],
]

# Conta quantas vezes dois produtos aparecem juntos no mesmo pedido.
pair_counts = Counter()
for _, products_in_order in cooc_source.groupby("order_id")["product_id"]:
    basket = sorted(set(products_in_order))
    if len(basket) < 2 or len(basket) > 30:
        continue
    for left, right in combinations(basket, 2):
        pair_counts[(left, right)] += 1
        pair_counts[(right, left)] += 1

# Mantém os produtos mais relacionados a cada produto de origem.
related_by_product = defaultdict(list)
for (product_id, related_product_id), count in pair_counts.items():
    related_by_product[product_id].append((related_product_id, count))

related_products = pd.DataFrame(
    [
        (product_id, related_product_id, count)
        for product_id, related_items in related_by_product.items()
        for related_product_id, count in sorted(related_items, key=lambda item: item[1], reverse=True)[:TOP_RELATED_PER_PRODUCT]
    ],
    columns=["seed_product_id", "product_id", "cooccurrence_score"],
)

# Usa os produtos mais fortes do histórico do usuário como sementes de recomendação.
seed_products = (
    user_product_features.sort_values(["user_id", "purchase_count", "last_order_number"], ascending=[True, False, False])
    .assign(seed_rank=lambda df: df.groupby("user_id").cumcount() + 1)
    .query("seed_rank <= @USER_SEED_PRODUCTS")
    [["user_id", "product_id"]]
    .rename(columns={"product_id": "seed_product_id"})
)

# Gera candidatos relacionados aos produtos que o usuário costuma comprar.
cooccurrence_candidates = (
    seed_products.merge(related_products, on="seed_product_id", how="inner")
    .groupby(["user_id", "product_id"], as_index=False)
    .agg(cooccurrence_score=("cooccurrence_score", "max"))
    .sort_values(["user_id", "cooccurrence_score"], ascending=[True, False])
    .assign(rank=lambda df: df.groupby("user_id").cumcount() + 1)
    .query("rank <= @USER_COOCCURRENCE_CANDIDATES")
    [["user_id", "product_id", "cooccurrence_score"]]
)
cooccurrence_candidates = with_source_flags(cooccurrence_candidates, "cooccurrence")

In [ ]:
# Junta popularidade do produto com sua categoria.
product_popularity = product_features[["product_id", "product_total_orders"]].merge(
    products[["product_id", "aisle_id", "department_id"]], on="product_id", how="left"
)

# Seleciona produtos populares por departamento.
top_products_by_department = (
    product_popularity.sort_values(["department_id", "product_total_orders"], ascending=[True, False])
    .assign(rank=lambda df: df.groupby("department_id").cumcount() + 1)
    .query("rank <= @CATEGORY_TOP_PRODUCTS")
    [["department_id", "product_id", "product_total_orders"]]
    .rename(columns={"product_total_orders": "category_popularity_score"})
)

# Seleciona produtos populares por corredor.
top_products_by_aisle = (
    product_popularity.sort_values(["aisle_id", "product_total_orders"], ascending=[True, False])
    .assign(rank=lambda df: df.groupby("aisle_id").cumcount() + 1)
    .query("rank <= @CATEGORY_TOP_PRODUCTS")
    [["aisle_id", "product_id", "product_total_orders"]]
    .rename(columns={"product_total_orders": "category_popularity_score"})
)

# Recomenda produtos populares nos departamentos favoritos do usuário.
favorite_department_candidates = (
    favorite_departments.query("rank <= @USER_FAVORITE_DEPARTMENTS")[["user_id", "department_id"]]
    .merge(top_products_by_department, on="department_id", how="inner")
    [["user_id", "product_id", "category_popularity_score"]]
)

# Recomenda produtos populares nos corredores favoritos do usuário.
favorite_aisle_candidates = (
    favorite_aisles.query("rank <= @USER_FAVORITE_AISLES")[["user_id", "aisle_id"]]
    .merge(top_products_by_aisle, on="aisle_id", how="inner")
    [["user_id", "product_id", "category_popularity_score"]]
)

# Consolida candidatos de departamento e corredor, mantendo o maior score.
category_candidates = (
    pd.concat([favorite_department_candidates, favorite_aisle_candidates], ignore_index=True)
    .groupby(["user_id", "product_id"], as_index=False)
    .agg(category_popularity_score=("category_popularity_score", "max"))
    .sort_values(["user_id", "category_popularity_score"], ascending=[True, False])
    .assign(rank=lambda df: df.groupby("user_id").cumcount() + 1)
    .query("rank <= @USER_CATEGORY_CANDIDATES")
    [["user_id", "product_id", "category_popularity_score"]]
)
category_candidates = with_source_flags(category_candidates, "favorite_category")

In [ ]:
# Agrupa usuários por corredor favorito para criar uma aproximação simples de similaridade.
similar_source = dataset_all_orders[["user_id", "product_id"]].merge(
    user_profile[["user_id", "favorite_aisle_id"]].dropna(), on="user_id", how="inner"
)

# Para cada grupo de usuários parecidos, seleciona os produtos mais comuns.
similar_top_products = (
    similar_source.groupby(["favorite_aisle_id", "product_id"], as_index=False)
    .agg(similar_user_score=("user_id", "nunique"))
    .sort_values(["favorite_aisle_id", "similar_user_score"], ascending=[True, False])
    .assign(rank=lambda df: df.groupby("favorite_aisle_id").cumcount() + 1)
    .query("rank <= @SIMILAR_TOP_PRODUCTS")
    [["favorite_aisle_id", "product_id", "similar_user_score"]]
)

# Entrega esses produtos como candidatos para usuários do mesmo grupo de interesse.
similar_candidates = (
    user_profile[["user_id", "favorite_aisle_id"]]
    .merge(similar_top_products, on="favorite_aisle_id", how="inner")
    [["user_id", "product_id", "similar_user_score"]]
)
similar_candidates = with_source_flags(similar_candidates, "similar_users")

### Consolidação, target e gravação no SQLite

Os produtos do conjunto `train` são usados para marcar o `target`. Também são incluídos na lista de candidatos para que a base supervisionada contenha exemplos positivos, mas nenhuma flag de origem usa essa informação futura.

In [ ]:
# Inclui produtos positivos do conjunto train para permitir aprendizado supervisionado.
# As flags ficam zeradas para não usar informação futura como origem da recomendação.
target_candidates = train_user_products.copy()
for column in [
    "candidate_from_history",
    "candidate_from_cooccurrence",
    "candidate_from_favorite_category",
    "candidate_from_similar_users",
]:
    target_candidates[column] = 0
for score_column in ["cooccurrence_score", "category_popularity_score", "similar_user_score"]:
    target_candidates[score_column] = 0.0

candidate_columns = [
    "user_id",
    "product_id",
    "candidate_from_history",
    "candidate_from_cooccurrence",
    "candidate_from_favorite_category",
    "candidate_from_similar_users",
    "cooccurrence_score",
    "category_popularity_score",
    "similar_user_score",
]

# Consolida todas as fontes em um único par usuário-produto.
candidates = (
    pd.concat(
        [
            history_candidates[candidate_columns],
            cooccurrence_candidates[candidate_columns],
            category_candidates[candidate_columns],
            similar_candidates[candidate_columns],
            target_candidates[candidate_columns],
        ],
        ignore_index=True,
    )
    .groupby(["user_id", "product_id"], as_index=False)
    .agg(
        candidate_from_history=("candidate_from_history", "max"),
        candidate_from_cooccurrence=("candidate_from_cooccurrence", "max"),
        candidate_from_favorite_category=("candidate_from_favorite_category", "max"),
        candidate_from_similar_users=("candidate_from_similar_users", "max"),
        cooccurrence_score=("cooccurrence_score", "max"),
        category_popularity_score=("category_popularity_score", "max"),
        similar_user_score=("similar_user_score", "max"),
    )
)

# Junta candidatos com features de usuário, produto, categoria e histórico.
training_data = (
    candidates.merge(user_product_features, on=["user_id", "product_id"], how="left")
    .merge(user_features, on="user_id", how="left")
    .merge(product_features, on="product_id", how="left")
    .merge(product_catalog[["product_id", "aisle_id", "department_id", "aisle", "department"]], on="product_id", how="left")
    .merge(user_department_counts, on=["user_id", "department_id"], how="left")
    .merge(user_aisle_counts, on=["user_id", "aisle_id"], how="left")
    .merge(user_profile, on="user_id", how="left")
)

In [ ]:
# Marca se o candidato já foi comprado pelo usuário ou se é um produto novo para ele.
training_data["candidate_was_previously_purchased"] = training_data["purchase_count"].notna().astype("int8")
training_data["candidate_is_new_product_for_user"] = (1 - training_data["candidate_was_previously_purchased"]).astype("int8")

# Produtos novos não possuem histórico usuário-produto; nesses casos usamos zero.
for column in [
    "purchase_count",
    "reorder_rate",
    "avg_cart_position",
    "first_order_number",
    "last_order_number",
    "user_department_purchase_count",
    "user_department_purchase_rate",
    "user_aisle_purchase_count",
    "user_aisle_purchase_rate",
]:
    training_data[column] = training_data[column].fillna(0)

# Cria variáveis derivadas para recência, frequência e afinidade de categoria.
training_data["orders_since_last_purchase"] = (
    training_data["user_total_orders"] - training_data["last_order_number"]
).where(training_data["candidate_was_previously_purchased"].eq(1), 0)
training_data["purchase_frequency"] = training_data["purchase_count"] / training_data["user_total_orders"].clip(lower=1)
training_data["is_favorite_department"] = (
    training_data["department_id"] == training_data["favorite_department_id"]
).fillna(False).astype("int8")
training_data["is_favorite_aisle"] = (
    training_data["aisle_id"] == training_data["favorite_aisle_id"]
).fillna(False).astype("int8")

# Define o alvo: 1 quando o usuário comprou o produto no conjunto train.
train_set = train_user_products.assign(target=1)
training_data = training_data.merge(train_set, on=["user_id", "product_id"], how="left")
training_data["target"] = training_data["target"].fillna(0).astype("int8")

training_data.shape

In [ ]:
# Seleciona somente as colunas que serão usadas no protótipo de modelagem.
final_columns = [
    "user_id", "product_id",
    "candidate_from_history", "candidate_from_cooccurrence", "candidate_from_favorite_category", "candidate_from_similar_users",
    "candidate_was_previously_purchased", "candidate_is_new_product_for_user",
    "cooccurrence_score", "category_popularity_score", "similar_user_score",
    "purchase_count", "reorder_rate", "avg_cart_position", "first_order_number", "last_order_number",
    "orders_since_last_purchase", "purchase_frequency",
    "user_total_orders", "user_total_items", "user_unique_products", "user_reorder_rate",
    "user_avg_cart_position", "user_avg_days_between_orders", "user_avg_order_hour", "user_avg_basket_size",
    "product_total_orders", "product_unique_users", "product_reorder_rate", "product_avg_cart_position",
    "aisle_id", "department_id", "aisle", "department",
    "user_department_purchase_count", "user_department_purchase_rate", "user_aisle_purchase_count", "user_aisle_purchase_rate",
    "is_favorite_department", "is_favorite_aisle", "target",
]
training_data = training_data[final_columns]

# Resume o tamanho da base e a participação de cada fonte de candidatos.
training_data_summary = pd.DataFrame(
    [
        {
            "rows": len(training_data),
            "columns": training_data.shape[1],
            "target_rate": training_data["target"].mean(),
            "new_candidate_rate": training_data["candidate_is_new_product_for_user"].mean(),
            "history_candidate_rate": training_data["candidate_from_history"].mean(),
            "cooccurrence_candidate_rate": training_data["candidate_from_cooccurrence"].mean(),
            "favorite_category_candidate_rate": training_data["candidate_from_favorite_category"].mean(),
            "similar_users_candidate_rate": training_data["candidate_from_similar_users"].mean(),
        }
    ]
)

training_data_summary

In [ ]:
# Escreve primeiro em uma tabela temporária para evitar perder a base antiga se algo falhar.
with sqlite3.connect(DATABASE_PATH) as conn:
    conn.execute(f"DROP TABLE IF EXISTS {NEW_TABLE_NAME}")
    training_data.to_sql(NEW_TABLE_NAME, conn, if_exists="replace", index=False, chunksize=100_000)

    # Substitui a tabela final somente depois que a nova escrita foi concluída.
    conn.execute(f"DROP TABLE IF EXISTS {TABLE_NAME}")
    conn.execute(f"ALTER TABLE {NEW_TABLE_NAME} RENAME TO {TABLE_NAME}")

    # Índices simples aceleram consultas por usuário, produto e alvo.
    conn.execute(f"CREATE INDEX IF NOT EXISTS idx_{TABLE_NAME}_user_product ON {TABLE_NAME} (user_id, product_id)")
    conn.execute(f"CREATE INDEX IF NOT EXISTS idx_{TABLE_NAME}_target ON {TABLE_NAME} (target)")

print(f"Tabela `{TABLE_NAME}` recriada em: {DATABASE_PATH}")

## 11. Conclusão da análise exploratória

A análise confirmou que o dataset permite construir uma recomendação baseada em comportamento, indo além de recompra. A nova base combina histórico individual, preferências por categoria, coocorrência de produtos, popularidade dentro dos interesses do usuário e sinais de usuários semelhantes.

Com isso, o próximo notebook pode focar em limpeza, normalização e validação da base preparada antes do treinamento.